> 在线讲义：[第 04 章 · 系统设计的关键数字：容量、算力、带宽、延迟](https://wikieden.github.io/ai-infra-from-constraints/lessons/04-key-numbers.html) · 本书《从约束推导 AI Infra》

# 第 4 章 · 系统设计的关键数字：容量、算力、带宽、延迟

配套讲义：仓库里 `ai-infra/notes/lessons/04-key-numbers.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 寻道 ÷ 主存 | $10^7 / 10^2$ | `DEAN_NS` | 100000 倍 |
| 第二步 寻道例子 | 20 × 10 + 1 | `seek_example()` | 201 → 200.5 / 101 ms |
| 第二步 AI 编程工具 | Amdahl | `amdahl(0.2, 5)` … `amdahl(0.8, 5)` | 1.19、1.32、2.27、2.78 |
| 第三步 读 1 GB | $1\ \mathrm{GB}/\beta$ | `read_ms(1e9, gpu)` | 0.99、0.49、0.30 ms |
| 第四步 70B BF16 | $N b_W$ | `weight_gb(70_553_706_496, 2)` | 141.11 GB = 131.42 GiB |
| 第四步 容量预算 | 三项相加 | `budget_gb(73.73, 2.5, 2)` | 78.56 GB |
| 第五步 网卡 | ÷ 8 | `gbit_to_gbyte(400)` | 50 GB/s，1 GB 要 20 ms |

数字取自 AII 第 1 章 §1.2：Jeff Dean 2009 年的 12 个操作耗时、三张 GPU 的速查表、DeepSeek-R1-Distill-Llama-70B 的参数数与量化结果。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
DEAN_NS = {
    "访问一级缓存": 0.5,
    "分支预测错误后的恢复": 5,
    "访问二级缓存": 7,
    "对互斥锁加锁或解锁": 25,
    "访问主存": 100,
    "用 Zippy 压缩 1K 字节": 3_000,
    "经 1 Gbit/s 网络发送 2K 字节": 20_000,
    "从主存连续读取 1 MB": 250_000,
    "同一数据中心内往返": 500_000,
    "磁盘寻道": 10_000_000,
    "从磁盘连续读取 1 MB": 20_000_000,
    "数据包从加州到荷兰再返回": 150_000_000,
}
GPUS = {
    "RTX 4090": (24, 1.008, 165.2),
    "A100 80GB SXM": (80, 2.039, 312),
    "H100 SXM": (80, 3.35, 989.4),
}
GIB = 2**30 / 1e9  # 1 GiB 合多少 GB

**函数 `seek_example`**：串行寻道 + CPU 计算的总时间（ms）。

In [ ]:
def seek_example(seeks: int = 20, seek_ms: float = 10, cpu_ms: float = 1) -> float:
    """串行寻道 + CPU 计算的总时间（ms）。"""
    return seeks * seek_ms + cpu_ms

**函数 `amdahl`**：Amdahl 定律：比例 f 的部分加速 s 倍，总加速比 1 / ((1-f) + f/s)。

In [ ]:
def amdahl(f: float, s: float) -> float:
    """Amdahl 定律：比例 f 的部分加速 s 倍，总加速比 1 / ((1-f) + f/s)。"""
    return 1 / ((1 - f) + f / s)

**函数 `read_ms`**：按带宽峰值读 nbytes 字节要几毫秒：R / β。

In [ ]:
def read_ms(nbytes: float, gpu: str) -> float:
    """按带宽峰值读 nbytes 字节要几毫秒：R / β。"""
    return nbytes / (GPUS[gpu][1] * 1e12) * 1e3

**函数 `weight_gb`**：权重大小 M_W = N × b_W（十进制 GB）。

In [ ]:
def weight_gb(n_params: int, bytes_per: float) -> float:
    """权重大小 M_W = N × b_W（十进制 GB）。"""
    return n_params * bytes_per / 1e9

**函数 `budget_gb`**：容量预算：权重（GB）+ KV（GiB）+ 工作区（GiB），统一换成 GB 再相加。

In [ ]:
def budget_gb(weights_gb: float, kv_gib: float, work_gib: float) -> float:
    """容量预算：权重（GB）+ KV（GiB）+ 工作区（GiB），统一换成 GB 再相加。"""
    return weights_gb + kv_gib * GIB + work_gib * GIB

**函数 `gbit_to_gbyte`**：网速换算：Gbit/s ÷ 8 = GB/s。

In [ ]:
def gbit_to_gbyte(gbit_per_s: float) -> float:
    """网速换算：Gbit/s ÷ 8 = GB/s。"""
    return gbit_per_s / 8

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：数量级（Jeff Dean 2009）")
print(f"  磁盘寻道 ÷ 访问主存 = {DEAN_NS['磁盘寻道'] / DEAN_NS['访问主存']:.0f} 倍")

In [ ]:
print("\n第二步：串行寻道的例子")
print(f"  原来 {seek_example():.1f} ms；CPU 快一倍 {seek_example(cpu_ms=0.5):.1f} ms；寻道减半 {seek_example(seeks=10):.1f} ms")
print("  AI 编程工具让编码快 5 倍：")
for f in (0.2, 0.3, 0.7, 0.8):
    print(f"    编码占 {f:.0%} → 总加速比 {amdahl(f, 5):.2f}")

In [ ]:
print("\n第三步：按带宽峰值读 1 GB")
for gpu in GPUS:
    print(f"  {gpu:14s} {read_ms(1e9, gpu):.2f} ms")

In [ ]:
print("\n第四步：DeepSeek-R1-Distill-Llama-70B")
w = weight_gb(70_553_706_496, 2)
print(f"  BF16 权重 {w:.2f} GB = {w / GIB:.2f} GiB；均分两张卡每张 {w / 2:.2f} GB，剩 {80 - w / 2:.2f} GB")
print(f"  8 比特方案 73.73 GB，一张 H100 剩 {80 - 73.73:.2f} GB")
print(f"  容量预算 73.73 + 2.5 GiB + 2 GiB = {budget_gb(73.73, 2.5, 2):.2f} GB ≤ 80 GB")

In [ ]:
print("\n第五步：网卡 400 Gbit/s")
bw = gbit_to_gbyte(400)
print(f"  = {bw:.0f} GB/s，按线速传 1 GB 要 {1 / bw * 1e3:.0f} ms")

## 3. 代码里才看得到的

1. `budget_gb` 里 KV 和工作区是按 GiB 给的，权重是 GB；不换算直接相加会少算约 0.33 GB，正好吃掉一部分余量。
2. `amdahl(f, float("inf"))` 在 Python 里能直接算出上限 $1/(1-f)$，因为 `f / inf == 0`。
3. 原书的 70.55、9.45 等数都是四舍五入后的两位小数，测试里用 `round(…, 2)` 比较。

## 4. 自己改一改

- 把 `amdahl` 的 `s` 改成 `float("inf")`，看 $f = 0.2$ 和 $f = 0.8$ 的上限分别是多少。
- 用 `read_ms` 算一算三张卡各读 16.38 GB（Qwen3-8B 的权重）要多久。
- 把 `budget_gb` 的 KV 从 2.5 GiB 加倍，看还放不放得下。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 先看量级再动手，十万倍差别一眼瞅；
> 加速一段看占比，剩下不动是上限；
> 放得下看容量，耗多久看算力和带宽；
> 参数乘字节算权重，bit 除以八才是 byte。